# Tech Addiction Prediction: Neural Network (MLP)
In this notebook, we build a deep Neural Network using `TensorFlow/Keras`. 
Neural Networks capture different, smooth non-linear relationships compared to gradient boosted trees, providing crucial diversity for our final ensemble.


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)


## 1. Data Loading


In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
SUBMISSION_PATH = '/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv'

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

X = train_df.drop(['id', 'addicted_label'], axis=1)
y = train_df['addicted_label'].values


## 2. V5 Feature Engineering & Scaling
Neural Networks are highly sensitive to the scale of input features. We must `StandardScale` everything.


In [ ]:
def engineer_features(train, test):
    train = train.copy()
    test = test.copy()
    
    for df in [train, test]:
        # Original V2 Features
        df['weekend_delta'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
        df['social_media_prop'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['unaccounted_screen_time'] = df['daily_screen_time_hours'] - (df['social_media_hours'] + df['gaming_hours'] + df['work_study_hours'])
        
        # NEW V5 Features
        df['productivity_ratio'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['entertainment_ratio'] = (df['social_media_hours'] + df['gaming_hours']) / (df['daily_screen_time_hours'] + 1e-5)
        df['screen_to_sleep_ratio'] = df['daily_screen_time_hours'] / (df['sleep_hours'] + 1e-5)
        df['interaction_intensity'] = df['notifications_per_day'] * df['app_opens_per_day']
        df['sleep_deprived'] = (df['sleep_hours'] < 6.5).astype(int)
        df['age_group'] = pd.cut(df['age'], bins=[0, 20, 30, 40, 50, 100], labels=False)
        
    return train, test

print("Engineering Deep V5 features...")
X, test_df_eng = engineer_features(X, test_df.drop(['id'], axis=1))

categorical_features = ['gender', 'academic_work_impact', 'stress_level', 'age_group']
numeric_features = [col for col in X.columns if col not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(test_df_eng)
print(f"Processed Train shape: {X_processed.shape}")


## 3. Architecture Definition


In [ ]:
def build_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='swish'),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(64, activation='swish'),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(32, activation='swish'),
        BatchNormalization(),
        Dropout(0.2),
        
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')]
    )
    return model


## 4. Stratified 5-Fold Cross Validation


In [ ]:
print("Starting 5-Fold Stratified Cross-Validation for Neural Network...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds_proba = np.zeros(len(X_test_processed))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_train, X_val = X_processed[train_idx], X_processed[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = build_model(X_processed.shape[1])
    
    callbacks = [
        EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=5, min_lr=1e-5, verbose=1)
    ]
    
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=512,
        callbacks=callbacks,
        verbose=0 
    )
    
    # Predict OOF
    oof_preds[val_idx] = model.predict(X_val, batch_size=1024, verbose=0).flatten()
    
    # Predict Test
    test_preds_proba += model.predict(X_test_processed, batch_size=1024, verbose=0).flatten() / 5

print("\nCross-Validation complete!")


### Metric Evaluation


In [ ]:
print("Evaluating OOF predictions...")
roc_auc = roc_auc_score(y, oof_preds)
pr_auc = average_precision_score(y, oof_preds)

print("-" * 30)
print("Neural Network (5-Fold OOF) Performance:")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("-" * 30)


## 5. Submission


In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': test_preds_proba
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
display(submission.head())
